What is CoOrdinate descent 
update oe weight at a time, keeping others fixed 

w1 -> w2 -> w3 -> ... -> wm -> repeat 



Soft thresholding 

For each coefficient wj : 
$$w_j = \frac{S(\rho _j , \lambda)}{z_j}$$
where : 
 $$\rho _j = \sum x_{ij}(y_i -\hat{y}_i + w_j x_{ij}) $$
 $$z_j = \sum x_{ij} ^ 2$$

 $$S(\rho , \lambda) = if (\rho > \lambda) then \rho - \lambda $$
 $$ if (\rho < -\lambda) then \rho + \lambda $$
 $$if (| \rho | \le \lambda) then 0 $$ 

In [1]:
# Data 
import numpy as np 
np.random.seed(42) 
x = np.linspace(-3, 3, 100) 
y = 0.5 * x**3 - x**2 + x + np.random.randn(100)*3 

x_train, x_test = x[:70], x[70:] 
y_train, y_test = y[:70], y[70:] 


#polynomial features 
def polynomial_features(x, degree):
    return np.vstack([x**i for i in range(degree+1)]).T 

def standardize(x):
    x_scaled = x.copy()
    mean = x[:, 1:].mean(axis = 0) 
    std = x[:, 1:].std(axis = 0) 
    x_scaled[:, 1:] = (x[:, 1:]-mean)/std 
    return x_scaled 


# soft-thresholding function 
def soft_threshold(rho, lam):
    if rho > lam:
        return rho - lam 
    elif rho < -lam :
        return rho + lam 
    else : 
        return 0.0 

def lasso_coordinate_descent(x, y, lam, epochs=100):
    n, m = x.shape
    w = np.zeros(m) 
    for _ in range(epochs) :
        for j in range(1, m):
            y_pred = x @ w 
            rho = np.sum(x[:, j]*(y - y_pred + w[j]* x[:, j]))
            z = np.sum(x[:, j]**2)
            w[j] = soft_threshold(rho, lam)/z 

        #update bias x_0 separately 
        w[0] = np.mean(y-x[:, 1:] @ w[1:]) 

    return w 



degree = 10 
lam = 0.1 
xtr = polynomial_features(x_train, degree) 
xte = polynomial_features(x_test, degree) 

xtr = standardize(xtr) 
xte = standardize(xte) 

w = lasso_coordinate_descent(xtr, y_train, lam) 

train_error = np.mean((y_train - xtr @ w)**2)
test_error = np.mean((y_test - xte @ w)**2) 

print("weights:", w) 
print("Train Error: ", train_error) 
print("Test Error: ", test_error) 







weights: [-6.05970037  2.3502912  -3.58593686  2.06473646  0.2005848  -0.18724258
  0.47847107 -0.18530366  0.          0.03834643 -0.42353178]
Train Error:  6.5574931105871075
Test Error:  89.1619383631295
